# Part D — Experiment 1: Inspect the Sparse Representation

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

In [2]:
# 7.1 Lọc dữ liệu
df = pd.read_json('c4-train.00000-of-01024-30K.json.gz', lines=True, compression='gzip')
corpus = df['text'].tolist()

In [3]:
# 7.1 Lọc dữ liệu
df = pd.read_json('c4-train.00000-of-01024-30K.json.gz', lines=True, compression='gzip')
corpus = df['text'].tolist() 

# 7.2 Xây dựng pipeline
# 1. Tokenizer & CountVectorizer -> TF Matrix
vectorizer = CountVectorizer()
X_counts = vectorizer.fit_transform(corpus) 

# 2. TF-IDF Transformer -> TF-IDF Matrix
tfidf_transformer = TfidfTransformer()
X_tfidf = tfidf_transformer.fit_transform(X_counts) 

# 7.3 Kiểm tra kích thước
N, V = X_tfidf.shape
print(f"Number of documents (N) = {N}")
print(f"Vocabulary size (V) = {V}")
print(f"Matrix shape = {X_tfidf.shape}")

# 7.4 Kiểm tra sparsity
nnz = X_tfidf.nnz
sparsity = 1 - (nnz / (N * V))
print(f"Sparsity (S) = {sparsity:.6f}")

# 7.5 Inspect vocabulary
feature_names = np.array(vectorizer.get_feature_names_out())

# Top 20 terms phổ biến nhất theo DF
df_counts = np.array((X_counts > 0).sum(axis=0)).flatten()
top_20_df_indices = df_counts.argsort()[-20:][::-1]
print("\nTop 20 terms theo DF:\n", feature_names[top_20_df_indices])

# Top 20 terms có IDF cao nhất
idf_scores = tfidf_transformer.idf_
top_20_idf_indices = idf_scores.argsort()[-20:][::-1]
print("\nTop 20 terms có IDF cao nhất:\n", feature_names[top_20_idf_indices])

# Top 20 terms có TF-IDF cao nhất trong document đầu tiên (index = 0)
doc_index = 0
doc_tfidf_scores = X_tfidf[doc_index].toarray().flatten()
top_20_tfidf_indices = doc_tfidf_scores.argsort()[-20:][::-1]
print(f"\nTop 20 terms có TF-IDF cao nhất trong document {doc_index}:\n", feature_names[top_20_tfidf_indices])

Number of documents (N) = 30000
Vocabulary size (V) = 193540
Matrix shape = (30000, 193540)
Sparsity (S) = 0.999141

Top 20 terms theo DF:
 ['the' 'and' 'to' 'of' 'in' 'for' 'is' 'with' 'on' 'that' 'this' 'are'
 'it' 'as' 'at' 'from' 'be' 'you' 'by' 'have']

Top 20 terms có IDF cao nhất:
 ['00000' '00003' '000040' '00005' '0000856166' '0001042' '000116' '00012'
 '00015' '00016' '000165101' '0002' '00022' '000226' '000281' '확인하게'
 '환원되지' '활동' '활동에' '활동을']

Top 20 terms có TF-IDF cao nhất trong document 0:
 ['bbq' 'class' 'meat' 'balay' 'kcbs' 'lonestar' 'will' 'missoula' 'apron'
 'smoker' 'you' 'timelines' 'trimming' 'spectators' 'cost' 'rangers'
 '22nd' 'beginner' 'beginners' 'culinary']


## Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều V?

Để đảm bảo toàn về số chiều của toàn bộ vector, thực hiện các phép toán ma trận


## Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?

Không, do công thức TF-IDF còn dựa vào DF nữa, IDF cao thì DF thấp => TF-IDF không cao

# Part F — Experiment 2: Preprocessing Ablation

In [4]:
import pandas as pd
import string
import time
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# 1. Tải dữ liệu và chia tập Train / Test (để tính OOV)
df = pd.read_json('c4-train.00000-of-01024-30K.json.gz', lines=True, compression='gzip')
corpus = df['text'].tolist()

train_corpus = corpus[:5000] # Tập từ vựng
test_corpus = corpus[5000:6000] # Tập để đo OOV

stop_words = set(stopwords.words('english'))
subword_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Định nghĩa các Pipelines
def pipeline_a(text):
    return text.lower().split()

def pipeline_b(text):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return [t for t in text.split() if t not in stop_words]

def pipeline_c(text):
    return subword_tokenizer.tokenize(text)

# 3. Hàm đo lường Metrics
def evaluate(pipeline_func, name):
    print(f"\n--- Running {name} ---")
    
    # Preprocessing
    train_tokens = [pipeline_func(doc) for doc in train_corpus]
    test_tokens = [pipeline_func(doc) for doc in test_corpus]
    
    # Vectorize
    vectorizer = CountVectorizer(analyzer=lambda x: x)
    X_train = vectorizer.fit_transform(train_tokens)
    
    vocab = vectorizer.vocabulary_
    vocab_set = set(vocab.keys())
    
    # Tính Metrics
    vocab_size = len(vocab)
    avg_tokens = sum(len(doc) for doc in train_tokens) / len(train_tokens)
    
    N, V = X_train.shape
    sparsity = 1 - (X_train.nnz / (N * V))
    
    # OOV Rate trên tập test
    test_vocab = set(token for doc in test_tokens for token in doc)
    oov_tokens = test_vocab - vocab_set
    oov_rate = len(oov_tokens) / len(test_vocab) if len(test_vocab) > 0 else 0
    
    # Search Performance (thời gian search 1 query)
    query_tokens = pipeline_func("artificial intelligence and machine learning in modern applications")
    query_vec = vectorizer.transform([query_tokens])
    
    start_time = time.time()
    _ = cosine_similarity(query_vec, X_train)
    search_time = time.time() - start_time
    
    print(f"Vocabulary size: {vocab_size}")
    print(f"Average tokens/document: {avg_tokens:.2f}")
    print(f"Matrix sparsity: {sparsity:.6f}")
    print(f"OOV rate: {oov_rate*100:.2f}%")
    print(f"Search performance: {search_time:.5f} seconds")

# 4. Thực thi
evaluate(pipeline_a, "Pipeline A (Minimal)")
evaluate(pipeline_b, "Pipeline B (Normalized)")
evaluate(pipeline_c, "Pipeline C (Extended)")

c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warn


--- Running Pipeline A (Minimal) ---
Vocabulary size: 140104
Average tokens/document: 373.10
Matrix sparsity: 0.998657
OOV rate: 37.34%
Search performance: 0.07113 seconds

--- Running Pipeline B (Normalized) ---


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2531 > 512). Running this sequence through the model will result in indexing errors


Vocabulary size: 85645
Average tokens/document: 213.15
Matrix sparsity: 0.998385
OOV rate: 32.94%
Search performance: 0.05616 seconds

--- Running Pipeline C (Extended) ---
Vocabulary size: 25828
Average tokens/document: 477.83
Matrix sparsity: 0.992396
OOV rate: 2.38%
Search performance: 0.06105 seconds


# Part G — Application: Build a Document Search Engine

In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class DocumentSearchEngine:
    def __init__(self, corpus):
        self.corpus = corpus
        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)
    def search(self, query, top_k=5):
        """
        Thực hiện tìm kiếm với User Query.
        Pipeline: User Query -> TF-IDF Query Vector -> Cosine Similarity -> Ranking -> Top-K
        """
        # 1. Chuyển đổi query thành TF-IDF vector
        query_vector = self.vectorizer.transform([query])
        
        # 2. Tính Cosine Similarity giữa query vector và toàn bộ documents
        similarities = cosine_similarity(query_vector, self.tfidf_matrix).flatten()
        
        # 3. Ranking: Lấy ra top-K tài liệu có độ tương đồng cao nhất
        top_k_indices = similarities.argsort()[-top_k:][::-1]
        
        # 4. Trích xuất thông tin hiển thị
        results = []
        for rank, doc_id in enumerate(top_k_indices, start=1):
            sim_score = similarities[doc_id]
            
            # Bỏ qua các tài liệu có độ tương đồng = 0
            if sim_score == 0:
                continue
                
            # Tạo preview tài liệu
            doc_content = self.corpus[doc_id].replace('\n', ' ')
            preview = doc_content[:80] + "..." if len(doc_content) > 80 else doc_content
            
            results.append({
                "Rank": rank,
                "Document ID": doc_id,
                "Similarity": round(sim_score, 4),
                "Document preview": preview
            })
            
        return pd.DataFrame(results)

if __name__ == "__main__":
    mock_corpus = [
        "A novel approach to medical image classification using convolutional neural networks.",
        "Natural language processing has seen immense progress with the advent of large models.",
        "The transformer language model architecture is the foundation of modern NLP tasks.",
        "Deep learning is revolutionizing healthcare by providing accurate diagnostic tools.",
        "An overview of traditional machine learning algorithms versus deep learning approaches.",
        "Transformer-based models for processing electronic health records in healthcare.",
        "Image segmentation and classification for medical diagnostics using deep learning.",
        "Advances in natural language processing for automated translation systems."
    ]

    # Khởi tạo Search Engine
    search_engine = DocumentSearchEngine(mock_corpus)

    # Các query test theo yêu cầu
    queries = [
        "medical image classification",
        "transformer language model",
        "deep learning healthcare",
        "natural language processing"
    ]

   # Thực hiện search và in kết quả
    for q in queries:
        print(f"**Query:** \"{q}\"")
        df_results = search_engine.search(q, top_k=5)
        
        # In ra DataFrame mặc định thay vì dùng to_markdown
        print(df_results) 
        


**Query:** "medical image classification"
   Rank  Document ID  Similarity  \
0     1            6      0.5344   
1     2            0      0.4891   

                                    Document preview  
0  Image segmentation and classification for medi...  
1  A novel approach to medical image classificati...  
**Query:** "transformer language model"
   Rank  Document ID  Similarity  \
0     1            2      0.4446   
1     2            5      0.1680   
2     3            7      0.1328   
3     4            1      0.1073   

                                    Document preview  
0  The transformer language model architecture is...  
1  Transformer-based models for processing electr...  
2  Advances in natural language processing for au...  
3  Natural language processing has seen immense p...  
**Query:** "deep learning healthcare"
   Rank  Document ID  Similarity  \
0     1            3      0.4549   
1     2            4      0.3727   
2     3            6      0.2912   
3     

# Part H — Evaluation 

## CÁC HÀM TÍNH TOÁN METRIC ĐÁNH GIÁ

In [6]:
def precision_at_k(retrieved_docs, relevant_docs, k=5):
    """Tính Precision@K: Số doc liên quan tìm được trong Top K / K"""
    retrieved_k = retrieved_docs[:k]
    relevant_retrieved = set(retrieved_k).intersection(set(relevant_docs))
    return len(relevant_retrieved) / k

def recall_at_k(retrieved_docs, relevant_docs, k=5):
    """Tính Recall@K: Số doc liên quan tìm được trong Top K / Tổng số doc liên quan"""
    retrieved_k = retrieved_docs[:k]
    relevant_retrieved = set(retrieved_k).intersection(set(relevant_docs))
    if len(relevant_docs) == 0:
        return 0.0
    return len(relevant_retrieved) / len(relevant_docs)

def reciprocal_rank(retrieved_docs, relevant_docs):
    """Tính Reciprocal Rank cho 1 query: 1 / rank của doc liên quan đầu tiên"""
    for rank, doc_id in enumerate(retrieved_docs, start=1):
        if doc_id in relevant_docs:
            return 1.0 / rank
    return 0.0

## CHẠY PIPELINE ĐÁNH GIÁ & LƯU KẾT QUẢ

In [7]:
if __name__ == "__main__":
    mock_corpus = [
            "A novel approach to medical image classification using convolutional neural networks.",
            "Natural language processing has seen immense progress with the advent of large models.",
            "The transformer language model architecture is the foundation of modern NLP tasks.",
            "Deep learning is revolutionizing healthcare by providing accurate diagnostic tools.",
            "An overview of traditional machine learning algorithms versus deep learning approaches.",
            "Transformer-based models for processing electronic health records in healthcare.",
            "Image segmentation and classification for medical diagnostics using deep learning.",
            "Advances in natural language processing for automated translation systems."
    ]

    search_engine = DocumentSearchEngine(mock_corpus)

    # Tập đánh giá
    # Định nghĩa sẵn các document IDs thực sự Relevant cho mỗi query
    evaluation_set = {
        "medical image classification": [0, 6],
        "transformer language model": [2, 5],
        "deep learning healthcare": [3, 4, 5, 6],
        "natural language processing": [1, 2, 7]
    }

    # Danh sách lưu kết quả
    evaluation_results = []
    
    K = 5
    mrr_sum = 0.0

    # Chạy đánh giá cho từng query
    for query, relevant_docs in evaluation_set.items():
        # Lấy danh sách doc ID trả về từ hệ thống
        retrieved_docs = search_engine.search(query, top_k=K)
        
        # Tính toán metrics
        p_at_5 = precision_at_k(retrieved_docs, relevant_docs, k=K)
        r_at_5 = recall_at_k(retrieved_docs, relevant_docs, k=K)
        rr = reciprocal_rank(retrieved_docs, relevant_docs)
        
        mrr_sum += rr
        
        # Ghi nhận kết quả
        evaluation_results.append({
            "Query": query,
            "Relevant Docs (Ground Truth)": str(relevant_docs),
            "Retrieved Docs (Top 5)": str(retrieved_docs),
            "Precision@5": round(p_at_5, 4),
            "Recall@5": round(r_at_5, 4),
            "Reciprocal Rank": round(rr, 4)
        })

    # Đưa kết quả vào DataFrame
    df_results = pd.DataFrame(evaluation_results)

    # Tính Mean cho các metrics (MRR, Mean P@5, Mean R@5)
    mean_p5 = df_results["Precision@5"].mean()
    mean_r5 = df_results["Recall@5"].mean()
    mrr = mrr_sum / len(evaluation_set)

    # Thêm một dòng tổng hợp (Mean) vào cuối bảng
    mean_row = {
        "Query": "AVERAGE (MEAN)",
        "Relevant Docs (Ground Truth)": "-",
        "Retrieved Docs (Top 5)": "-",
        "Precision@5": round(mean_p5, 4),
        "Recall@5": round(mean_r5, 4),
        "Reciprocal Rank": round(mrr, 4)
    }
    df_results = pd.concat([df_results, pd.DataFrame([mean_row])], ignore_index=True)

    # Xuất ra file results.csv
    csv_filename = "results.csv"
    df_results.to_csv(csv_filename, index=False, encoding='utf-8')
    
    print(f"Đã hoàn thành đánh giá. Kết quả được lưu tại: {csv_filename}")
    print("\nBảng xem trước kết quả:")
    print(df_results)

Đã hoàn thành đánh giá. Kết quả được lưu tại: results.csv

Bảng xem trước kết quả:
                          Query Relevant Docs (Ground Truth)  \
0  medical image classification                       [0, 6]   
1    transformer language model                       [2, 5]   
2      deep learning healthcare                 [3, 4, 5, 6]   
3   natural language processing                    [1, 2, 7]   
4                AVERAGE (MEAN)                            -   

                              Retrieved Docs (Top 5)  Precision@5  Recall@5  \
0     Rank  Document ID  Similarity  \\n0     1  ...          0.0       0.0   
1     Rank  Document ID  Similarity  \\n0     1  ...          0.0       0.0   
2     Rank  Document ID  Similarity  \\n0     1  ...          0.0       0.0   
3     Rank  Document ID  Similarity  \\n0     1  ...          0.0       0.0   
4                                                  -          0.0       0.0   

   Reciprocal Rank  
0              0.0  
1              